In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import glob
import shap
import json
import networkx as nx
import seaborn as sns
import lingam
import warnings
import requests

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler,
    MinMaxScaler,
)

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from typing import Dict
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score
from scipy.stats import pearsonr
from statsmodels.tsa.stattools import ccf
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.api import VAR
from scipy import stats
from dotenv import load_dotenv
from collections import defaultdict
from IPython.display import display, Markdown
from datetime import datetime

from sklearn.metrics import (
    roc_auc_score, 
    roc_curve, 
    confusion_matrix, 
    precision_score, 
    recall_score, 
    accuracy_score,
    classification_report
)


load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

warnings.filterwarnings('ignore')
print("✅ Imports loaded successfully!")

✅ Imports loaded successfully!


In [9]:
lag_features_path = f"../../../data/azure_pm/lag_features/machine_98_lag_features.csv"
df_machine = pd.read_csv(lag_features_path)
print(f"Loaded shape: {df_machine.shape}")
df_machine.head()

Loaded shape: (8757, 65)


,datetime,volt,rotate,pressure,vibration,errorID,comp,failure,target,volt_lag_1h,...,error_count_6h,error_count_24h,maint_count_6h,maint_count_24h,hour,day_of_week,is_weekend,is_working_hours,hours_since_maint,hours_since_error
0,2015-01-01 06:00:00,0.303137,0.604476,0.271993,0.621000,0,0,0,0,0.000000,...,0.0,0.0,0.0,0.0,6,3,0,0,0,0
1,2015-01-01 07:00:00,0.455312,0.635538,0.480756,0.645421,0,0,0,0,0.303137,...,0.0,0.0,0.0,0.0,7,3,0,0,1,1
2,2015-01-01 08:00:00,0.449470,0.540973,0.427196,0.356126,0,0,0,0,0.455312,...,0.0,0.0,0.0,0.0,8,3,0,1,2,2
3,2015-01-01 09:00:00,0.408248,0.687752,0.378658,0.254123,0,0,0,0,0.449470,...,0.0,0.0,0.0,0.0,9,3,0,1,3,3
4,2015-01-01 10:00:00,0.486041,0.518746,0.454206,0.351050,0,0,0,0,0.408248,...,0.0,0.0,0.0,0.0,10,3,0,1,4,4


In [40]:
def get_random_nonzero_target_row(df):
    filtered = df[(df["target"] != 0) & (df["failure"]==0)]
    if filtered.empty:
        return None
    return filtered.sample(n=1, random_state=np.random.randint(0, 10000)).iloc[0]
    # return filtered
    







# Example usage:
random_row = get_random_nonzero_target_row(df_machine)[["datetime","failure", "target"]]
# random_row_full = random_row.copy()
# random_row.drop(["datetime", "failure", "target"], inplace=True)

# # random_row.apply(pd.to_numeric)
# display(random_row.to_frame().T)

In [24]:
random_row

,datetime,failure,target
672,2015-01-29 06:00:00,0,3
673,2015-01-29 07:00:00,0,3
674,2015-01-29 08:00:00,0,3
675,2015-01-29 09:00:00,0,3
676,2015-01-29 10:00:00,0,3
...,...,...,...
7911,2015-11-26 02:00:00,0,3
7912,2015-11-26 03:00:00,0,3
7913,2015-11-26 04:00:00,0,3
7914,2015-11-26 05:00:00,0,3


In [41]:
random_row

datetime    2015-02-13 12:00:00
failure                       0
target                        1
Name: 1038, dtype: object

In [34]:
random_row

,datetime,failure,target
672,2015-01-29 06:00:00,0,3
673,2015-01-29 07:00:00,0,3
674,2015-01-29 08:00:00,0,3
675,2015-01-29 09:00:00,0,3
676,2015-01-29 10:00:00,0,3
...,...,...,...
7911,2015-11-26 02:00:00,0,3
7912,2015-11-26 03:00:00,0,3
7913,2015-11-26 04:00:00,0,3
7914,2015-11-26 05:00:00,0,3


In [13]:
# Check the failure and target variables 24 hours before a failure occures
df_machine_original = pd.read_csv(f"../../../data/azure_pm/machines/machine_98.csv")

display(df_machine_original.iloc[random_row_full.to_frame().T.index.values[0]-24:random_row_full.to_frame().T.index.values[0]+1][["datetime", "failure"]])
display(df_machine.iloc[random_row_full.to_frame().T.index.values[0]-24:random_row_full.to_frame().T.index.values[0]+1][["datetime","failure", "target"]])

,datetime,failure
5348,2015-08-11 13:00:00,0
5349,2015-08-11 14:00:00,0
5350,2015-08-11 15:00:00,0
5351,2015-08-11 16:00:00,0
5352,2015-08-11 17:00:00,0
5353,2015-08-11 18:00:00,0
5354,2015-08-11 19:00:00,0
5355,2015-08-11 20:00:00,0
5356,2015-08-11 21:00:00,0
5357,2015-08-11 22:00:00,0


,datetime,failure,target
5348,2015-08-11 13:00:00,0,0
5349,2015-08-11 14:00:00,0,0
5350,2015-08-11 15:00:00,0,0
5351,2015-08-11 16:00:00,0,0
5352,2015-08-11 17:00:00,0,0
5353,2015-08-11 18:00:00,0,0
5354,2015-08-11 19:00:00,0,0
5355,2015-08-11 20:00:00,0,0
5356,2015-08-11 21:00:00,0,0
5357,2015-08-11 22:00:00,0,0
